# 結晶検出（YOLO） - 学習・評価（Jupyter版）
`data/images/` + `data/labels/` + `data/data.yaml`（`colab/yolo_dataset_pipeline.ipynb`でエクスポートしたもの）
を使って、事前学習済みYOLOをファインチューニングする。`yolo_project/`フォルダを
カレントディレクトリとしてこのnotebookを実行すること。

> VSCodeで開く場合: 右上でカーネル（Python環境）を選択してから、上から順にセルを実行してください。

## 準備

In [ ]:
from pathlib import Path
from ultralytics import YOLO

DATA_YAML = "data/data.yaml"
MODEL = "yolov8n.pt"   # 事前学習済み・軽量モデル。精度が足りなければ yolov8s.pt / yolov8m.pt へ
EPOCHS = 100
IMGSZ = 640
BATCH = 16
NAME = "crystal_yolo"

assert Path(DATA_YAML).exists(), f"{DATA_YAML} が見つかりません。data/ にデータセットを展開してください。"
print("準備完了")

## （任意）大きい結晶のオーバーサンプリング
サイズ別の検出率チェックで、大きい結晶ほど検出率が低い場合に実行する。
大きい結晶を含むtrainパッチを複製して登場頻度を上げる（valデータには影響しない）。
再実行しても安全（前回作った複製ファイルを消してから作り直す）。

In [ ]:
import shutil
from PIL import Image

THRESHOLD_PX = 100  # この値(px)以上の矩形を含むパッチを「大きい」とみなす
FACTOR = 8           # 該当パッチを何倍に増やすか（1なら複製なし）

train_img_dir = Path(DATA_YAML).parent / "images" / "train"
train_lbl_dir = Path(DATA_YAML).parent / "labels" / "train"


def has_large_box(label_path, img_w, img_h, threshold_px):
    with open(label_path) as f:
        for line in f:
            parts = line.split()
            if len(parts) != 5:
                continue
            _, xc, yc, bw, bh = map(float, parts)
            size_px = max(bw * img_w, bh * img_h)
            if size_px >= threshold_px:
                return True
    return False


# 再実行しても安全なように、前回作った複製ファイルを先に削除
removed = 0
for p in list(train_img_dir.glob("*_dup*.png")) + list(train_lbl_dir.glob("*_dup*.txt")):
    p.unlink()
    removed += 1
if removed:
    print(f"前回の複製ファイルを削除: {removed}件")

original_images = sorted(train_img_dir.glob("*.png"))
targets = []
for img_path in original_images:
    lbl_path = train_lbl_dir / (img_path.stem + ".txt")
    if not lbl_path.exists():
        continue
    with Image.open(img_path) as im:
        w, h = im.size
    if has_large_box(lbl_path, w, h, THRESHOLD_PX):
        targets.append(img_path.stem)

print(f"元のtrainパッチ数: {len(original_images)}件")
print(f"大きい結晶(>={THRESHOLD_PX:.0f}px)を含むパッチ: {len(targets)}件")

for stem in targets:
    img_path = train_img_dir / f"{stem}.png"
    lbl_path = train_lbl_dir / f"{stem}.txt"
    for i in range(1, FACTOR):
        shutil.copy(img_path, train_img_dir / f"{stem}_dup{i}.png")
        shutil.copy(lbl_path, train_lbl_dir / f"{stem}_dup{i}.txt")

print(f"複製後のtrainパッチ数: {len(list(train_img_dir.glob('*.png')))}件")
print("\n※ valデータは変更していません（評価の公平性のため）")

## 学習

In [ ]:
model = YOLO(MODEL)
model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    name=NAME,
)
best_weights = model.trainer.save_dir / "weights" / "best.pt"
print(f"\n保存先: {best_weights}")

## 評価（mAP・Precision・Recall）

In [ ]:
eval_model = YOLO(str(best_weights))
metrics = eval_model.val(data=DATA_YAML, imgsz=IMGSZ)

print(f"mAP50    = {metrics.box.map50:.4f}")
print(f"mAP50-95 = {metrics.box.map:.4f}")
print(f"Precision = {metrics.box.mp:.4f}")
print(f"Recall    = {metrics.box.mr:.4f}")

## 予測結果を目視確認

In [ ]:
import random
import matplotlib.pyplot as plt

val_img_dir = Path(DATA_YAML).parent / "images" / "val"
all_val_images = list(val_img_dir.glob("*.png"))
sample_paths = random.sample(all_val_images, min(4, len(all_val_images)))

preds = eval_model.predict(sample_paths, conf=0.25, imgsz=IMGSZ)

fig, axes = plt.subplots(1, len(preds), figsize=(4 * len(preds), 4))
if len(preds) == 1:
    axes = [axes]
for ax, r in zip(axes, preds):
    ax.imshow(r.plot()[:, :, ::-1])  # BGR -> RGB
    ax.axis("off")
plt.tight_layout()
plt.show()